# Change a policy ACL

`PUT /admin/policies/{id}/acl` changes who can read the policy. The harness
keeps the body hash and the version, and moves `updated_at`. The connector
still writes a snapshot: the stored ACL differs, and the snapshot hash covers
the bytes and the ACL, so `kbs-content-hash` moves while `kbs-version` stays.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition and create


In [ ]:
POLICY_ID = "pol-demo-acl"
KEY = f"raw/hr/policy/{POLICY_ID}.md"
BODY = "# Demo policy\n\nCreated by the notebook.\n"
PAYLOAD = {
    "policy_id": POLICY_ID,
    "title": "Demo policy",
    "category": "leave",
    "region": "DE",
    "format": "markdown",
    "native_acl": ["hr_employee"],
    "body": BODY,
}
hr_demo.delete_if_present("policy", POLICY_ID)
created = hr_demo.admin("POST", "/admin/policies", PAYLOAD)
print(created["policy_id"], "version", created["version"], created["native_acl"])


In [ ]:
def ready() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    return (
        BODY.encode() in hr_demo.object_body(KEY)
        and meta.get(hr_demo.META_ACL) == "hr_employee"
        and meta.get(hr_demo.META_VERSION) == str(created["version"])
        and len(meta.get(hr_demo.META_HASH, "")) == 64
        and all(meta.get(name) for name in hr_demo.META_KEYS)
    )

hr_demo.wait_until(ready, hr_demo.WRITE_WAIT, "create snapshot")
print(hr_demo.snapshot(KEY)["Metadata"])


## ACL change


In [ ]:
before = hr_demo.admin("GET", f"/policies/{POLICY_ID}")
before_head = hr_demo.snapshot(KEY)["Metadata"]
changed = hr_demo.admin(
    "PUT",
    f"/admin/policies/{POLICY_ID}/acl",
    {"native_acl": ["hr_policy_admin"]},
)
assert changed["version"] == before["version"]
assert changed["content_hash"] == before["content_hash"]
assert changed["native_acl"] == ["hr_policy_admin"]

def acl_written() -> bool:
    head = hr_demo.snapshot(KEY)
    if head is None:
        return False
    meta = head["Metadata"]
    return (
        meta.get(hr_demo.META_ACL) == "hr_policy_admin"
        and meta.get(hr_demo.META_HASH) != before_head[hr_demo.META_HASH]
        and meta.get(hr_demo.META_VERSION) == before_head[hr_demo.META_VERSION]
    )

hr_demo.wait_until(acl_written, hr_demo.WRITE_WAIT, "acl snapshot")
print(hr_demo.snapshot(KEY)["Metadata"])


## Cleanup


In [ ]:
hr_demo.delete_if_present("policy", POLICY_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)
